# Feature Engineering & Data Preprocessing

## Objective

The objective of this notebook is to prepare the dataset for machine learning model development by:

- Removing non-informative features
- Creating train-test splits
- Classifying feature types
- Building a preprocessing pipeline
- Encoding categorical variables
- Scaling numerical variables
- Saving preprocessing artifacts for model training

In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import os
import joblib

In [41]:
df=pd.read_csv("data\\alzheimers_disease_data.csv")
df.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


## Load Dataset

The original dataset is loaded for feature engineering and preprocessing.

Insights from the EDA phase are used to guide preprocessing decisions, including feature removal, encoding, and scaling strategies.

In [42]:
df.drop(['PatientID','DoctorInCharge'],axis=1,inplace=True)

## Remove Non-Predictive Features

Based on findings from the EDA phase:

- PatientID is a unique identifier.
- DoctorInCharge contains administrative information.

These features do not contribute to prediction performance and are removed.

In [43]:
X = df.drop('Diagnosis', axis=1)
y = df['Diagnosis']

## Separate Features and Target

The target variable is separated from the feature set prior to preprocessing.

In [44]:
binary_cols = [col for col in X.columns if X[col].nunique() == 2]
categorical_feature_cols = ["Ethnicity","EducationLevel"]
continuous_cols = [col for col in X.columns if col not in binary_cols + categorical_feature_cols]   


In [45]:
print("Binary Features:")
print(binary_cols)

print("\nCategorical Features:")
print(categorical_feature_cols)

print("\nContinuous Features:")
print(continuous_cols)

Binary Features:
['Gender', 'Smoking', 'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes', 'Depression', 'HeadInjury', 'Hypertension', 'MemoryComplaints', 'BehavioralProblems', 'Confusion', 'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks', 'Forgetfulness']

Categorical Features:
['Ethnicity', 'EducationLevel']

Continuous Features:
['Age', 'BMI', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality', 'SystolicBP', 'DiastolicBP', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'ADL']


## Feature Type Identification

Features are categorized into:

- Continuous Features
- Binary Features
- Categorical Features

This classification determines the preprocessing strategy for each feature group.

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Train-Test Split

The dataset is divided into training and testing sets using stratified sampling to preserve the target distribution.

In [54]:
print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (1719, 32)
Test Shape: (430, 32)


### Observation

- The dataset was split into training and testing sets using an 80:20 ratio.
- Stratified sampling was applied to preserve the class distribution in both datasets.

In [47]:
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, continuous_cols),
        ('cat', categorical_transformer, categorical_feature_cols)
    ],
    remainder='passthrough'
)

## Build Preprocessing Pipeline

The preprocessing pipeline applies:

- Standard Scaling to continuous variables
- One-Hot Encoding to categorical variables
- Pass-through for binary variables

This ensures consistent transformations during training and inference.

In [48]:


os.makedirs("../artifacts", exist_ok=True)

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('../artifacts/train.csv', index=False)
test_df.to_csv('../artifacts/test.csv', index=False)

In [49]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [50]:
y_train.value_counts(normalize=True)

Diagnosis
0    0.646306
1    0.353694
Name: proportion, dtype: float64

In [51]:
feature_names = preprocessor.get_feature_names_out()

print("Total Features After Transformation:", len(feature_names))
print("Feature Names:", feature_names)

Total Features After Transformation: 38
Feature Names: ['num__Age' 'num__BMI' 'num__AlcoholConsumption' 'num__PhysicalActivity'
 'num__DietQuality' 'num__SleepQuality' 'num__SystolicBP'
 'num__DiastolicBP' 'num__CholesterolTotal' 'num__CholesterolLDL'
 'num__CholesterolHDL' 'num__CholesterolTriglycerides' 'num__MMSE'
 'num__FunctionalAssessment' 'num__ADL' 'cat__Ethnicity_0'
 'cat__Ethnicity_1' 'cat__Ethnicity_2' 'cat__Ethnicity_3'
 'cat__EducationLevel_0' 'cat__EducationLevel_1' 'cat__EducationLevel_2'
 'cat__EducationLevel_3' 'remainder__Gender' 'remainder__Smoking'
 'remainder__FamilyHistoryAlzheimers' 'remainder__CardiovascularDisease'
 'remainder__Diabetes' 'remainder__Depression' 'remainder__HeadInjury'
 'remainder__Hypertension' 'remainder__MemoryComplaints'
 'remainder__BehavioralProblems' 'remainder__Confusion'
 'remainder__Disorientation' 'remainder__PersonalityChanges'
 'remainder__DifficultyCompletingTasks' 'remainder__Forgetfulness']


In [52]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(1719, 38)
(430, 38)


### Observation

The preprocessing pipeline successfully transformed both training and testing datasets while maintaining consistent feature dimensions.

In [53]:


joblib.dump(
    preprocessor,
    '../artifacts/preprocessor.pkl'
)

['../artifacts/preprocessor.pkl']

## Save Preprocessing Artifact

The fitted preprocessing pipeline is saved for reuse during model training and deployment.

# Feature Engineering Summary

1. Removed non-predictive features (PatientID and DoctorInCharge).
2. Separated features and target variable.
3. Classified features into binary, categorical, and continuous groups.
4. Split data into training and testing sets using stratified sampling.
5. Applied One-Hot Encoding to categorical features.
6. Applied Standard Scaling to continuous features.
7. Preserved binary features without transformation.
8. Saved the fitted preprocessing pipeline for model training.

### Missing Value Handling

The dataset contains no missing values; therefore, no imputation strategy was required in the preprocessing pipeline.

### Future Experiment

EducationLevel is currently treated as a categorical feature using One-Hot Encoding.
During model experimentation, an alternative approach will be tested where EducationLevel is retained as an ordinal numerical feature.